# 📘 Agentic Architectures 19: LATS + MCTS (Language Agent Tree Search)

In this notebook we implement **Language Agent Tree Search (LATS)** — one of the most advanced architectures for complex reasoning tasks. It combines the capabilities of language models with the powerful **Monte Carlo Tree Search (MCTS)** search algorithm.

While Tree of Thoughts (ToT) simply explores options, LATS adds the ability to **learn from mistakes** within a single task. Using a self-critique (reflection) mechanism and backpropagation of the evaluation score, the agent can understand which branches of reasoning lead to a dead end and focus its efforts on promising directions.

### Definition
**LATS (Language Agent Tree Search)** is an architecture in which an LLM acts as an agent reasoning within a search space, while MCTS drives that search. It includes four key stages:
1. **Selection:** Choosing the most promising tree node based on UCB1 (Upper Confidence Bound).
2. **Expansion:** Generating new steps (thoughts) from the selected node.
3. **Evaluation/Reflection:** Scoring the quality of the step using the LLM (self-critique).
4. **Backpropagation:** Updating the value of all parent nodes based on the resulting score.

### Application: Game of 24
For the demonstration, we use the classic math puzzle "Game of 24." Given 4 numbers (e.g. `4, 9, 10, 13`), you must use the arithmetic operations `(+, -, *, /)` to get exactly 24. This is an excellent benchmark for testing planning and search abilities.

## Phase 0: Foundation and Setup

### Step 0.1: Installing libraries

**What we'll do:**
We'll install the necessary libraries. We'll need the standard LangChain and LangGraph tooling.

In [ ]:
# !pip install -q -U langchain-nebius langchain langgraph rich python-dotenv

### Step 0.2: Environment setup

**What we'll do:**
We'll configure access to the Nebius API and initialize the model. For this task it's better to use a temperature > 0 to generate a diverse set of candidate solutions (thoughts).

In [ ]:
import os
import math
import random
import re
from typing import List, Dict, Any, Optional, Tuple
from collections import deque
from dotenv import load_dotenv

# LangChain & Graph
from langchain_nebius import ChatNebius
from langgraph.graph import StateGraph, END
from pydantic import BaseModel, Field

# UI
from rich.console import Console
from rich.tree import Tree
from rich.panel import Panel

from notebook_utils import setup_environment

console = setup_environment(
    project_name="Agentic Architecture - LATS (Nebius)",
    required_keys=["NEBIUS_API_KEY", "LANGCHAIN_API_KEY"]
)

# Higher temperature for more creative candidate solutions
llm = ChatNebius(model="meta-llama/Meta-Llama-3.1-8B-Instruct", temperature=0.7)

### Step 0.3: The "Game of 24" environment

**What we'll do:**
We'll implement the game logic. We need a `Game24Env` class that:
1. Takes a string of numbers (e.g. "4 9 10 13").
2. Validates a proposed solution (expression validity and whether it equals 24).
3. Generates new tasks.

In [ ]:
class Game24Env:
    @staticmethod
    def validate_expression(numbers: str, expression: str) -> bool:
        """Exact, bounded + - * / arithmetic using each original number once."""
        import ast
        from fractions import Fraction
        from collections import Counter
        if len(expression) > 256:
            return False
        try:
            targets = [int(n) for n in numbers.split()]
            if len(targets) != 4:
                return False
            tree = ast.parse(expression.strip(), mode="eval")
            if len(list(ast.walk(tree))) > 64:
                return False
            used = []
            def calculate(node):
                if isinstance(node, ast.Constant) and type(node.value) is int:
                    used.append(node.value)
                    return Fraction(node.value)
                if isinstance(node, ast.BinOp) and type(node.op) in (ast.Add, ast.Sub, ast.Mult, ast.Div):
                    left, right = calculate(node.left), calculate(node.right)
                    if isinstance(node.op, ast.Add):
                        return left + right
                    if isinstance(node.op, ast.Sub):
                        return left - right
                    if isinstance(node.op, ast.Mult):
                        return left * right
                    return left / right
                raise ValueError("Only basic arithmetic is allowed")
            result = calculate(tree.body)
            return Counter(used) == Counter(targets) and result == 24
        except (ValueError, SyntaxError, ZeroDivisionError, OverflowError):
            return False

    @staticmethod
    def solution_expression(numbers: str, state: str):
        """Read a full expression; a partial operation or judge score is not proof."""
        import re
        match = re.search(r"solution:\s*([^\n]+)", state, re.IGNORECASE)
        expression = (match.group(1) if match else state).split("=")[0].strip()
        if expression.startswith("op:"):
            expression = expression[3:].strip()
        return expression if Game24Env.validate_expression(numbers, expression) else None

print("Game24 environment defined.")


## Phase 1: Baseline — Tree of Thoughts (ToT)

As a baseline for comparison, we'll use a simplified version of Tree of Thoughts. It simply generates N candidate solutions and checks them, without smart search or error backpropagation.

### Step 1.1: Implementing ToT

**What we'll do:**
1. Create a `solve_tot` function that asks the LLM to generate 5 different expressions.
2. Validate each one via `Game24Env`.
3. If a correct solution is found, return success.

In [ ]:
async def solve_tot(numbers: str, n_thoughts: int = 5) -> str:
    console.print(f"--- [ToT] Solving for: {numbers} ---")

    prompt = f"""
    Use numbers {numbers} and basic arithmetic operations (+ - * /) to obtain 24.
    Each number must be used exactly once.
    Output only the mathematical expression, nothing else.
    Examples:
    Input: 4 4 6 8
    Output: (4 + 8) * (6 - 4)
    Input: 2 9 10 12
    Output: 2 * 12 * (10 - 9)
    """

    # Generate N candidates in parallel or sequentially
    candidates = []
    for _ in range(n_thoughts):
        res = llm.invoke(prompt).content.strip()
        candidates.append(res)

    for expr in candidates:
        if Game24Env.validate_expression(numbers, expr):
            return f"SUCCESS: {expr}"

    return "FAILURE: No valid solution found"

print("Baseline ToT defined.")

## Phase 2: The advanced approach — LATS with MCTS

A proper LATS implementation requires managing a search tree.

### Step 2.1: The MCTS tree-node class

**What we'll do:**
We'll define an `MCTSNode` class that stores:
- `state`: The current state (a partial or complete solution).
- `parent`: The parent node.
- `children`: The child nodes.
- `visits`: How many times this path has been visited.
- `value`: The accumulated value (quality score).
- `reflection`: Text self-critique/analysis of this step.

We'll also implement an `upper_confidence_bound` method for selecting the best node (balancing exploitation and exploration).

In [ ]:
class MCTSNode:
    def __init__(self, state: str, parent=None, reflection: str = ""):
        self.state = state  # E.g.: "4 + 9 = 13. Left: 10, 13, 13"
        self.parent = parent
        self.children: List[MCTSNode] = []
        self.visits = 0
        self.value = 0.0
        self.reflection = reflection

    def uct_score(self, exploration_weight: float = 1.0) -> float:
        """Upper Confidence Bound for Trees (UCT)."""
        if self.visits == 0:
            return float('inf')

        # Average value (exploitation)
        q_value = self.value / self.visits
        # Novelty factor (exploration)
        u_value = exploration_weight * math.sqrt(math.log(self.parent.visits) / self.visits)

        return q_value + u_value

    def add_child(self, child_state: str, reflection: str):
        child = MCTSNode(child_state, parent=self, reflection=reflection)
        self.children.append(child)
        return child

    def backpropagate(self, reward: float):
        """Updates this node's statistics and all its ancestors'."""
        self.visits += 1
        self.value += reward
        if self.parent:
            self.parent.backpropagate(reward)

    def best_child(self):
        """Selects the child node with the highest UCT."""
        if not self.children:
            return None
        return max(self.children, key=lambda c: c.uct_score())

### Step 2.2: Implementing the LATS Agent

**What we'll do:**
We'll implement the `LATSAgent` class with the main loop:
1. **Select:** Starting from the root, descend through the best child nodes (Select) until reaching a leaf.
2. **Expand:** Use the LLM to generate several (k) possible continuations from this node.
3. **Evaluate/Reflect:** For each continuation, ask the LLM to score its quality (0.0 - 1.0) and write a reflection.
4. **Backpropagate:** Propagate the resulting score back up the tree.
5. **Repeat:** Repeat the loop N times or until a solution is found.

In [ ]:
class LATSAgent:
    def __init__(self, numbers: str, simulations: int = 5):
        self.root = MCTSNode(state=f"Start numbers: {numbers}")
        self.root.visits = 1  # To avoid log(0) at the root
        self.numbers = numbers
        self.simulations = simulations

    def solve(self) -> str:
        for i in range(self.simulations):
            console.print(f"\n[bold blue]--- Iteration {i+1}/{self.simulations} ---[/bold blue]")

            # 1. Selection
            node = self.select_node(self.root)
            console.print(f"Selected node: {node.state}")

            # Check whether the state is already a final solution
            expression = Game24Env.solution_expression(self.numbers, node.state)
            if expression is not None:
                return self.format_solution(node, expression)

            # 2. Expansion
            candidates = self.expand_node(node)

            # 3. Evaluation & Backpropagation
            for cand_state in candidates:
                # Evaluation and reflection
                reward, reflection = self.evaluate_and_reflect(cand_state)
                console.print(f"Candidate: [cyan]{cand_state}[/cyan] -> Reward: {reward:.2f}")
                # Create a new node
                child = node.add_child(cand_state, reflection)
                # Propagate the reward
                child.backpropagate(reward)

                expression = Game24Env.solution_expression(self.numbers, cand_state)
                if expression is not None:
                    return self.format_solution(child, expression)

        # After all simulations, look for the best path
        return self.get_best_solution()

    def select_node(self, node: MCTSNode) -> MCTSNode:
        """Greedy descent through the tree based on UCT."""
        current = node
        while current.children:
            current = current.best_child()
        return current

    def expand_node(self, node: MCTSNode, k: int = 3) -> List[str]:
        """Generates K candidate next steps."""
        prompt = f"""
        Original numbers: {self.numbers}
        Trajectory: {self.path_states(node)}
        Current state: {node.state}
        Target: Make 24 using remaining numbers.

        Propose {k} diverse next possible steps. A step is ONE arithmetic operation between two numbers.
        Output a list of operations formatted as: 'op: <expression> = <result>, remaining: <nums>'.
        When complete, include 'solution: <full expression using every original number exactly once>'.
        Example: 'op: 4 + 8 = 12, remaining: 12, 6'
        """
        response = llm.invoke(prompt).content
        # Simple line parsing
        lines = [line.strip() for line in response.split('\n') if "op:" in line]
        return lines[:k] if lines else [response] # Fallback

    def evaluate_and_reflect(self, state: str) -> Tuple[float, str]:
        """The LLM scores the state (0.0-1.0) and writes a reflection."""
        prompt = f"""
        Evaluate this step towards solving Game of 24: "{state}"

        1. Is it a valid math operation?
        2. Are we getting closer to 24 or stuck?
        3. If it reached 24 exactly, score is 1.0

        Output JSON: {{"score": 0.0 to 1.0, "reflection": "analysis"}}
        """
        try:
            res = llm.invoke(prompt).content
            import json
            match = re.search(r'\{.*\}', res, re.DOTALL)
            if match:
                data = json.loads(match.group())
                return float(data.get("score", 0.0)), data.get("reflection", "")
        except:
            pass
        return 0.1, "Error evaluating"

    @staticmethod
    def path_states(node):
        path = []
        while node is not None:
            path.append(node.state)
            node = node.parent
        return list(reversed(path))

    def format_solution(self, node, expression):
        return f"FOUND: {expression}\nPath: " + " -> ".join(self.path_states(node))

    def get_best_solution(self) -> str:
        return "FAILURE: No independently verified solution found within the search budget"

print("LATS Agent class defined.")

## Phase 3: Demonstration and Comparison

### Step 3.1: Testing on a hard example

**What we'll do:**
We'll run LATS on the set of numbers `4 9 10 13`, which is considered fairly hard for simple agents.

In [ ]:
tasks = ["4 9 10 13"]

# 1. Baseline ToT
print("Running Baseline ToT...")
# await solve_tot(tasks[0]) # Uncomment to run

# 2. LATS
console.print("\n[bold green]Running LATS Agent...[/bold green]")
agent = LATSAgent(tasks[0], simulations=6)
result = agent.solve()
console.print(Panel(result, title="LATS Result", border_style="green"))

## Conclusion

LATS demonstrates how **tree search** combined with **self-reflection** lets an agent find solutions to complex problems where linear reasoning often hits a dead end. The backpropagation mechanism lets the agent "learn" to avoid bad paths in the future, within the same session.